# Benchpress-Style Workouts — SF vs. Qiskit

Provider-agnostic benchmarking using the Superfermion benchmarks module.
This notebook runs IBM Benchpress-inspired workouts across multiple SDK strategies.

**Workouts covered:**
- **Construction** (8 tests): Circuit building, parameter binding, QASM import
- **Manipulation** (4 tests): Pauli twirling, gate decomposition, basis translation
- **Transpilation** (9 tests): Full compilation to hardware topology

**Methodology:**
- Each workout runs 3 rounds (configurable)
- Mean wall-clock time reported
- Quality metrics: 2Q gate count, circuit depth after transpilation

In [ ]:
from superfermion.benchmarks import BenchmarkRunner, TopologyFactory, list_strategies
from superfermion.benchmarks.strategies import SuperfermionStrategy

print(f"Available strategies: {list_strategies()}")

## Setup

Configure the target hardware topology and SDK strategies.

In [ ]:
# Target backend — heavy-hex topology (127Q) with ECR basis
backend = TopologyFactory.create(
    "heavy_hex",
    n_qubits=127,
    basis_gates=["rz", "sx", "x", "ecr"],
)
print(f"Backend: {backend.n_qubits}Q, {len(backend.coupling_map)} edges")
print(f"Basis: {backend.basis_gates}")

# SDK strategies
strategies = [SuperfermionStrategy()]

# Uncomment to include Qiskit (requires qiskit installed):
# from superfermion.benchmarks.strategies import QiskitStrategy
# strategies.append(QiskitStrategy())

print(f"Strategies: {[s.name for s in strategies]}")

## 1. Construction Workouts

Measure circuit building speed: QV, DTC, Clifford, MCX, SU2, QASM import.

In [ ]:
runner = BenchmarkRunner()

construction_report = runner.run(
    strategies=strategies,
    rounds=3,
    category="construction",
)
print()
print(construction_report.summary_table())

## 2. Manipulation Workouts

Test circuit transformation: Pauli twirling, decomposition, basis change.

In [ ]:
manipulation_report = runner.run(
    strategies=strategies,
    rounds=3,
    category="manipulation",
)
print()
print(manipulation_report.summary_table())

## 3. Transpilation Workouts

End-to-end compilation to hardware topology. Uses the heavy-hex 127Q backend.

**Note:** These can take 10-60s+ per test on large circuits. Use `all_to_all` for faster runs.

In [ ]:
# For quick runs, use all-to-all (no routing).
# For realistic benchmarking, use heavy_hex.
transpile_backend = TopologyFactory.create(
    "all_to_all",
    n_qubits=100,
    basis_gates=["rz", "sx", "x", "cx"],
)

transpilation_report = runner.run(
    strategies=strategies,
    rounds=1,
    category="transpilation",
    backend=transpile_backend,
)
print()
print(transpilation_report.summary_table())

## 4. Visualization & Export

In [ ]:
# Merge all reports for a combined view
from superfermion.benchmarks.runner import RunnerReport

full_report = RunnerReport()
for r in construction_report.results:
    full_report.add(r)
for r in manipulation_report.results:
    full_report.add(r)
for r in transpilation_report.results:
    full_report.add(r)

print(f"Total benchmarks: {len(full_report.results)}")

In [ ]:
# Generate bar chart
full_report.plot(title="Benchpress Workouts — Superfermion")

In [ ]:
# Export to pytest-benchmark-compatible JSON
json_path = "../benchmark_results.json"
full_report.to_json(json_path)
print(f"Results exported to {json_path}")

In [ ]:
# Speedup table (useful when multiple strategies are compared)
if len(strategies) > 1:
    print(full_report.speedup_table())
else:
    print("Single strategy — speedup comparison requires 2+ SDKs.")
    print("Uncomment QiskitStrategy in setup cell to compare.")

In [ ]:
# Quality metrics
from superfermion.benchmarks.report import generate_quality_table
print(generate_quality_table(full_report))